# LyreVoice — Colab Training

Train the LyreVoice voice cloning GAN on a T4 GPU.

**Prerequisites:** Upload these 3 tar.gz archives to `My Drive/lyrevoice/`:
- `preprocessed.tar.gz` — mel spectrograms + metadata
- `vctk_wavs.tar.gz` — raw VCTK wavs for speaker encoder
- `pretrained.tar.gz` — Tacotron2 + HiFi-GAN checkpoints

Create them locally with:
```bash
cd /path/to/lyrevoice
tar czf /tmp/preprocessed.tar.gz -C data preprocessed/
tar czf /tmp/vctk_wavs.tar.gz -C data/datasets/VCTK-Corpus wav48_silence_trimmed/
tar czf /tmp/pretrained.tar.gz -C checkpoints pretrained/
# Then upload the 3 files from /tmp/ to Google Drive: My Drive/lyrevoice/
```

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Clone the repo
!git clone https://github.com/greemwahr/lyrevoice.git /content/lyrevoice
%cd /content/lyrevoice

In [ ]:
# Install dependencies (pip, not uv — Colab doesn't have uv)
!pip install torch torchaudio --quiet
!pip install resemblyzer librosa soundfile pyyaml wandb gradio \
    frechet-audio-distance numpy scipy tqdm matplotlib setuptools --quiet

In [ ]:
# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected! Change runtime to T4 GPU.")

In [ ]:
# Extract archives from Drive to Colab local storage (much faster I/O)
import os

drive_root = '/content/drive/MyDrive/lyrevoice'
local_root = '/content/lyrevoice_data'
os.makedirs(local_root, exist_ok=True)

archives = [
    ('preprocessed.tar.gz', local_root),
    ('vctk_wavs.tar.gz', f'{local_root}/datasets/VCTK-Corpus'),
    ('pretrained.tar.gz', f'{local_root}'),
]

for archive, dest in archives:
    src = f'{drive_root}/{archive}'
    if not os.path.exists(src):
        print(f'MISSING: {src}')
        continue
    os.makedirs(dest, exist_ok=True)
    print(f'Extracting {archive}...')
    !tar xzf {src} -C {dest}
    print(f'  Done.')

print('\nExtraction complete. Verifying...')

# Verify extracted data
checks = [
    ('Preprocessed VCTK', f'{local_root}/preprocessed/vctk_metadata.txt'),
    ('Preprocessed LJSpeech', f'{local_root}/preprocessed/ljspeech_metadata.txt'),
    ('VCTK wavs', f'{local_root}/datasets/VCTK-Corpus/wav48_silence_trimmed'),
    ('Tacotron2 checkpoint', f'{local_root}/pretrained/tacotron2_statedict.pt'),
    ('HiFi-GAN checkpoint', f'{local_root}/pretrained/hifigan_generator.pt'),
    ('HiFi-GAN config', f'{local_root}/pretrained/hifigan_config.json'),
]
all_ok = True
for name, path in checks:
    exists = os.path.exists(path)
    status = 'OK' if exists else 'MISSING'
    print(f'  [{status}] {name}')
    if not exists:
        all_ok = False
if all_ok:
    print('\nAll data found! Ready to train.')
else:
    print('\nSome data is missing. Check your archives.')

In [ ]:
# Fix WandB server bug (flags: None)
import json
_orig = json.loads
def _safe(s, *a, **kw):
    if s is None:
        return {}
    return _orig(s, *a, **kw)
json.loads = _safe

# Login to WandB
import wandb
wandb.login()

In [ ]:
# Train with Colab config overlay
!python scripts/train.py \
    --config configs/config.yaml \
    --config-override configs/config_colab.yaml

In [ ]:
# Resume training after Colab disconnect
# First, find the latest checkpoint:
!ls -lhrt /content/drive/MyDrive/lyrevoice/checkpoints/ 2>/dev/null || echo 'No checkpoints found yet'

# Uncomment and set the checkpoint path to resume:
# !python scripts/train.py \
#     --config configs/config.yaml \
#     --config-override configs/config_colab.yaml \
#     --resume /content/drive/MyDrive/lyrevoice/checkpoints/lyrevoice_epoch_XXXX.pt